# Pre-Entrega 4 - Data Science II

## Extracción, transformación y persistencia de datos desde una API pública

**Caso de estudio:** Análisis descriptivo de países de América  
**Alumna:** Milagros Belén Brucellaria

---

## Introducción

En este proyecto se desarrollará un proceso de extracción de datos desde
una API REST pública utilizando Python.

Se utilizará una API de información geográfica que permite obtener datos
estructurados sobre diferentes países, como nombre, capital, región,
población y superficie.

A partir de los datos obtenidos se realizará un proceso básico de
extracción, transformación y persistencia, generando finalmente un
archivo CSV estructurado.

Además del cumplimiento técnico de la consigna, se realizará una breve
exploración de los datos para comprender la información recuperada.

## Objetivo general

Desarrollar un script en Python capaz de realizar una petición GET a una
API pública, procesar la respuesta obtenida, aplicar transformaciones
básicas y almacenar los resultados en un archivo CSV, incorporando
variables de entorno y manejo de errores.

## Pregunta de análisis

¿Cómo se distribuyen los países y territorios de América en términos de
población y superficie?

# 1. Preparación del entorno

Antes de realizar la extracción se preparará el entorno de trabajo.

Para este proyecto se utilizarán las siguientes librerías:

- **requests:** permite realizar peticiones HTTP desde Python.
- **pandas:** permite estructurar, transformar y analizar los datos.
- **python-dotenv:** permite cargar variables de entorno desde un archivo `.env`.
- **os:** permite acceder desde Python a las variables de entorno.

Separar las distintas responsabilidades facilita la comprensión y
mantenimiento del código.

#Instalación de librerias

In [3]:
!pip install python-dotenv -q

In [4]:
import os
import requests
import pandas as pd

from dotenv import load_dotenv

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


# 2. Fuente de datos

Para este proyecto se utiliza una API REST pública que proporciona
información estructurada sobre países.

Una API permite que dos aplicaciones intercambien información mediante
solicitudes. En este caso, Python actuará como cliente y solicitará
información a un servidor externo.

Se utilizará el método HTTP **GET**, ya que el objetivo es recuperar
información y no crear, modificar o eliminar recursos.

La respuesta proporcionada por la API será procesada en formato JSON.

# 3. Primera prueba de conexión

Antes de construir el proceso completo de extracción, se realizará una
petición simple.

El objetivo de esta prueba es verificar:

1. Que exista comunicación con la API.
2. Que el recurso solicitado esté disponible.
3. Qué código HTTP devuelve el servidor.
4. Qué estructura poseen los datos obtenidos.

Esta exploración inicial permite comprender la fuente antes de comenzar
la transformación de los datos.

In [5]:
url_prueba = "https://countries.dev/alpha/AR"

response_prueba = requests.get(
    url_prueba,
    timeout=10
)

print("Código HTTP:", response_prueba.status_code)

Código HTTP: 200


## Interpretación del código de estado

Los códigos de estado HTTP permiten conocer el resultado de una petición.

Un código de la familia 2xx representa generalmente una respuesta
exitosa, mientras que códigos de las familias 4xx y 5xx indican
situaciones de error del cliente o del servidor.

Validar este código es importante porque una aplicación no debería asumir
que una petición a un servicio externo siempre será exitosa.

In [6]:
data_prueba = response_prueba.json()

print("Tipo de objeto recibido:")
print(type(data_prueba))

print("\nContenido de la respuesta:")
data_prueba

Tipo de objeto recibido:
<class 'dict'>

Contenido de la respuesta:


{'area': 2780400,
 'cioc': 'ARG',
 'flag': '🇦🇷',
 'gini': 42.9,
 'name': 'Argentina',
 'flags': {'png': 'https://flagcdn.com/w320/ar.png',
  'svg': 'https://flagcdn.com/ar.svg'},
 'latlng': [-34, -64],
 'region': 'Americas',
 'borders': ['BOL', 'BRA', 'CHL', 'PRY', 'URY'],
 'capital': 'Buenos Aires',
 'demonym': 'Argentinean',
 'languages': [{'name': 'Spanish',
   'iso639_1': 'es',
   'iso639_2': 'spa',
   'nativeName': 'Español'},
  {'name': 'Guaraní',
   'iso639_1': 'gn',
   'iso639_2': 'grn',
   'nativeName': "Avañe'ẽ"}],
 'subregion': 'South America',
 'timezones': ['UTC-03:00'],
 'alpha2Code': 'AR',
 'alpha3Code': 'ARG',
 'currencies': [{'code': 'ARS', 'name': 'Argentine peso', 'symbol': '$'}],
 'nativeName': 'Argentina',
 'population': 45376763,
 'independent': True,
 'numericCode': '032',
 'callingCodes': ['54'],
 'topLevelDomain': ['.ar'],
 'populationDensity': 16.32}

### Interpretación de la respuesta obtenida

La petición realizada permitió recuperar correctamente información
estructurada sobre Argentina.

Al convertir la respuesta JSON mediante el método `.json()`, Python
representó el resultado como un objeto de tipo `dict` (diccionario).
Esto significa que la información se encuentra organizada mediante
pares clave-valor.

La inspección inicial permite observar variables simples, como
`name`, `capital`, `population` y `area`, y también estructuras más
complejas o anidadas, como `languages`, `currencies`, `flags` y
`borders`.

Esta exploración es importante antes de transformar los datos, ya que
permite conocer la estructura real de la fuente y determinar qué
variables serán relevantes para construir posteriormente el dataset
de análisis.

En este caso se identifican variables geográficas, categóricas y
numéricas que podrán ser seleccionadas en la etapa de transformación.

# 4. Extracción del conjunto de países

Luego de comprobar mediante una observación individual que la API
responde correctamente y de conocer la estructura de sus datos, el
siguiente paso consiste en ampliar la consulta.

El objetivo será recuperar el conjunto de países disponible en la API
para posteriormente seleccionar las observaciones y variables que
formarán parte del dataset final.

Esta estrategia permite separar una primera etapa exploratoria de la
extracción que posteriormente será utilizada para el procesamiento
de los datos.

In [7]:
url_paises = "https://countries.dev/countries"

response_paises = requests.get(
    url_paises,
    timeout=10
)

print("Código HTTP:", response_paises.status_code)

Código HTTP: 200


In [8]:
raw_data_prueba = response_paises.json()

print("Tipo de estructura:", type(raw_data_prueba))
print("Cantidad de registros recibidos:", len(raw_data_prueba))

Tipo de estructura: <class 'list'>
Cantidad de registros recibidos: 250


In [9]:
raw_data_prueba[0]

{'area': 652230,
 'cioc': 'AFG',
 'flag': '🇦🇫',
 'maps': {'googleMaps': 'https://www.google.com/maps/search/?api=1&query=33,65',
  'openStreetMaps': 'https://www.openstreetmap.org/?mlat=33&mlon=65#map=5/33/65'},
 'name': 'Afghanistan',
 'flags': {'png': 'https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/Flag_of_the_Taliban.svg/320px-Flag_of_the_Taliban.svg.png',
  'svg': 'https://upload.wikimedia.org/wikipedia/commons/5/5c/Flag_of_the_Taliban.svg'},
 'latlng': [33, 65],
 'region': 'Asia',
 'borders': ['IRN', 'PAK', 'TKM', 'UZB', 'TJK', 'CHN'],
 'capital': 'Kabul',
 'demonym': 'Afghan',
 'languages': [{'name': 'Pashto',
   'iso639_1': 'ps',
   'iso639_2': 'pus',
   'nativeName': 'پښتو'},
  {'name': 'Uzbek',
   'iso639_1': 'uz',
   'iso639_2': 'uzb',
   'nativeName': 'Oʻzbek'},
  {'name': 'Turkmen',
   'iso639_1': 'tk',
   'iso639_2': 'tuk',
   'nativeName': 'Türkmen'}],
 'subregion': 'Southern Asia',
 'timezones': ['UTC+04:30'],
 'alpha2Code': 'AF',
 'alpha3Code': 'AFG',
 'curre

### Inspección de una observación del conjunto de datos

Para comprender la estructura del conjunto de datos se inspeccionó la
primera observación recuperada mediante `raw_data_prueba[0]`.

Se observa que cada país contiene múltiples atributos. Algunos poseen
valores simples, como `name`, `region`, `population` y `area`, mientras
que otros contienen estructuras más complejas, como listas o
diccionarios.

Por ejemplo, variables como `languages`, `currencies`, `flags` y
`borders` presentan estructuras que requieren un tratamiento adicional
si se desean incorporar a una estructura tabular.

Esta inspección permite tomar una decisión fundamentada sobre qué
variables seleccionar para el dataset final, evitando conservar
información que no resulte necesaria para el objetivo del análisis.

In [10]:
print("Claves disponibles en cada registro:\n")

for clave in raw_data_prueba[0].keys():
    print("-", clave)


Claves disponibles en cada registro:

- area
- cioc
- flag
- maps
- name
- flags
- latlng
- region
- borders
- capital
- demonym
- languages
- subregion
- timezones
- alpha2Code
- alpha3Code
- currencies
- nativeName
- population
- independent
- numericCode
- altSpellings
- callingCodes
- translations
- regionalBlocs
- topLevelDomain
- populationDensity


### Selección de variables para el análisis

La exploración de las claves disponibles permitió identificar todas las
variables contenidas en cada observación.

La API proporciona más información de la necesaria para responder a la
pregunta planteada. Por este motivo se realizará una selección de
variables, conservando aquellas que permiten identificar cada país y
describir sus principales características geográficas y demográficas.

Las variables seleccionadas son:

- `name`: nombre del país.
- `alpha3Code`: código ISO de tres caracteres.
- `capital`: capital del país.
- `region`: región geográfica.
- `subregion`: subregión geográfica.
- `population`: población.
- `area`: superficie expresada en km².
- `populationDensity`: densidad poblacional.

Esta selección forma parte de la etapa de transformación, ya que permite
pasar de una respuesta JSON con múltiples atributos a una estructura más
simple y adecuada para el análisis.

In [11]:
pais_prueba = {
    "pais": raw_data_prueba[0].get("name"),
    "codigo_iso": raw_data_prueba[0].get("alpha3Code"),
    "capital": raw_data_prueba[0].get("capital"),
    "region": raw_data_prueba[0].get("region"),
    "subregion": raw_data_prueba[0].get("subregion"),
    "poblacion": raw_data_prueba[0].get("population"),
    "superficie_km2": raw_data_prueba[0].get("area"),
    "densidad_poblacional": raw_data_prueba[0].get("populationDensity")
}

pais_prueba

{'pais': 'Afghanistan',
 'codigo_iso': 'AFG',
 'capital': 'Kabul',
 'region': 'Asia',
 'subregion': 'Southern Asia',
 'poblacion': 40218234,
 'superficie_km2': 652230,
 'densidad_poblacional': 61.66}

### Interpretación de la transformación

La transformación de prueba se realizó correctamente.

A partir de la observación original, que contenía una gran cantidad de
atributos y estructuras anidadas, se generó un nuevo diccionario con
ocho variables seleccionadas para el análisis.

Además de seleccionar variables, se renombraron algunos atributos para
obtener nombres más claros y consistentes en el futuro dataset.

Por ejemplo:

- `name` se transformó en `pais`.
- `alpha3Code` se transformó en `codigo_iso`.
- `population` se transformó en `poblacion`.
- `area` se transformó en `superficie_km2`.
- `populationDensity` se transformó en `densidad_poblacional`.

Esta prueba permite validar la lógica de transformación sobre una única
observación antes de aplicarla al conjunto completo de datos.

También se utilizó el método `.get()` para acceder a los valores del
diccionario de manera controlada.

# 5. Gestión de configuración mediante variables de entorno

Una buena práctica de desarrollo consiste en separar los parámetros de
configuración de la lógica principal del programa.

Para este proyecto se utilizará un archivo `.env`.

La API utilizada no requiere credenciales de autenticación para las
consultas realizadas en este proyecto. Por este motivo, las variables
de entorno se utilizarán para almacenar parámetros de configuración y
no información sensible.

Se configurarán los siguientes parámetros:

- URL base de la API.
- Región que se desea analizar.
- Nombre del archivo de salida.
- Tiempo máximo de espera de una petición.

Esta separación permite modificar la configuración del proceso sin
necesidad de modificar la lógica del código.

Cuando el proyecto sea publicado en GitHub, el archivo `.env` será
excluido del repositorio mediante `.gitignore`. En su lugar se incluirá
un archivo `.env.example` como plantilla de configuración.

In [12]:
%%writefile .env
API_BASE_URL=https://countries.dev
REGION=Americas
OUTPUT_FILE=data_extracted.csv
TIMEOUT=10

Writing .env


In [13]:
load_dotenv(override=True)

API_BASE_URL = os.getenv("API_BASE_URL")
REGION = os.getenv("REGION")
OUTPUT_FILE = os.getenv("OUTPUT_FILE")
TIMEOUT = int(os.getenv("TIMEOUT"))

print("Configuración cargada correctamente.")
print("Región seleccionada:", REGION)
print("Archivo de salida:", OUTPUT_FILE)
print("Timeout:", TIMEOUT)

Configuración cargada correctamente.
Región seleccionada: Americas
Archivo de salida: data_extracted.csv
Timeout: 10


### Verificación de las variables de entorno

La configuración fue cargada correctamente mediante `python-dotenv`.

La función `load_dotenv()` permite leer las variables almacenadas en el
archivo `.env`, mientras que `os.getenv()` permite acceder a cada una de
ellas desde Python.

De esta manera, valores como la URL de la API, la región de análisis, el
archivo de salida y el timeout se encuentran separados de la lógica
principal del programa.

Esto permite evitar el hardcoding de parámetros de configuración y
facilita la reutilización del código en diferentes contextos.

# 6. Extracción de datos y manejo de errores

Hasta el momento se realizaron peticiones exploratorias para conocer el
funcionamiento de la API y la estructura de la respuesta.

A continuación se construirá una función específica para realizar la
extracción de los datos.

La función tendrá las siguientes responsabilidades:

1. Construir el endpoint utilizando la configuración definida.
2. Realizar una petición HTTP GET.
3. Establecer un tiempo máximo de espera.
4. Validar el estado HTTP de la respuesta.
5. Convertir la respuesta JSON a una estructura de Python.
6. Controlar posibles errores durante la comunicación con la API.

El manejo de excepciones es especialmente importante al trabajar con
fuentes externas, ya que existen situaciones que no dependen de nuestro
código, como problemas de conexión, demoras en el servidor o respuestas
HTTP no exitosas.

In [14]:
def extract_data():

    url = f"{API_BASE_URL}/countries"

    try:
        print("Iniciando extracción...")
        print("Endpoint:", url)

        response = requests.get(
            url,
            timeout=TIMEOUT
        )

        # Genera una excepción ante respuestas HTTP no exitosas
        response.raise_for_status()

        # Convierte la respuesta JSON a una estructura de Python
        data = response.json()

        print("Extracción realizada correctamente.")
        print("Registros recibidos:", len(data))

        return data

    except requests.exceptions.Timeout:
        print(
            "Error: la petición superó "
            "el tiempo máximo de espera."
        )

    except requests.exceptions.ConnectionError:
        print(
            "Error: no fue posible establecer "
            "conexión con la API."
        )

    except requests.exceptions.HTTPError as error:
        print(
            f"Error HTTP: {error}"
        )

    except requests.exceptions.RequestException as error:
        print(
            f"Error durante la petición: {error}"
        )

    except ValueError:
        print(
            "Error: la respuesta no contiene "
            "un JSON válido."
        )

    return None


### Manejo de excepciones

La petición se encuentra dentro de un bloque `try/except` para evitar
que una falla externa provoque una finalización abrupta del programa.

Se contemplan diferentes escenarios:

- `Timeout`: la API supera el tiempo máximo de espera establecido.
- `ConnectionError`: no es posible establecer conexión con el servidor.
- `HTTPError`: la API devuelve una respuesta HTTP no exitosa.
- `RequestException`: se produce otro error relacionado con la petición.
- `ValueError`: la respuesta obtenida no puede interpretarse como JSON.

También se utiliza `raise_for_status()` para validar el código HTTP
recibido y generar una excepción cuando la respuesta indica un error.

Esto mejora la robustez del proceso y permite informar claramente qué
tipo de problema ocurrió.

In [15]:
raw_data = extract_data()

Iniciando extracción...
Endpoint: https://countries.dev/countries
Extracción realizada correctamente.
Registros recibidos: 250


In [16]:
if raw_data is not None:

    print("Estado: extracción exitosa.")
    print("Tipo de estructura:", type(raw_data))
    print("Cantidad de registros:", len(raw_data))

else:

    print("Estado: no se pudieron obtener datos.")

Estado: extracción exitosa.
Tipo de estructura: <class 'list'>
Cantidad de registros: 250


# 7. Transformación del conjunto de datos

Una vez finalizada correctamente la extracción, el siguiente paso
consiste en transformar los datos crudos obtenidos desde la API.

La respuesta contiene 250 registros y múltiples atributos por país.
Sin embargo, para el objetivo de este proyecto no es necesario conservar
toda la información disponible.

La transformación tendrá como objetivos:

1. Seleccionar únicamente los registros correspondientes a América.
2. Conservar las variables relevantes para el análisis.
3. Renombrar las variables para mejorar su interpretación.
4. Convertir la información en una estructura tabular mediante Pandas.

Esta etapa permite pasar de los datos crudos obtenidos desde la API
a un dataset preparado para su posterior análisis.

In [17]:
regiones_disponibles = sorted(
    set(
        pais.get("region")
        for pais in raw_data
        if pais.get("region") is not None
    )
)

print("Regiones disponibles en la API:")

for region in regiones_disponibles:
    print("-", region)

Regiones disponibles en la API:
- Africa
- Americas
- Antarctic
- Antarctic Ocean
- Asia
- Europe
- Oceania
- Polar


### Interpretación de las categorías geográficas

Antes de aplicar el filtro se verificaron los valores únicos presentes
en la variable `region`.

La exploración confirmó que la fuente utiliza la categoría `Americas`
para identificar los registros correspondientes a América.

Esta comprobación es importante porque evita aplicar filtros basados en
suposiciones sobre los valores disponibles en la fuente.

Para obtener las categorías se utilizó `set()`, que permite eliminar
valores repetidos, y `sorted()`, que permite presentar los resultados
ordenados.

A partir de esta validación se utilizará el valor `Americas`, previamente
definido en la variable de entorno `REGION`, para seleccionar las
observaciones que formarán parte del dataset final.

# 7. Transformación del conjunto de datos

Una vez validada la categoría geográfica que se utilizará como filtro,
se procederá a transformar el conjunto completo de datos.

La transformación tendrá tres objetivos principales:

1. Filtrar únicamente las observaciones cuya región sea `Americas`.
2. Seleccionar las variables relevantes para el análisis.
3. Convertir los registros seleccionados en un DataFrame de Pandas.

De esta manera se pasará de una lista de diccionarios obtenida desde
la API a un dataset tabular preparado para su análisis.

In [18]:
def transform_data(data, region):

    registros = []

    for pais in data:

        # Seleccionamos únicamente la región configurada
        if pais.get("region") != region:
            continue

        # Seleccionamos y renombramos las variables relevantes
        registro = {
            "pais": pais.get("name"),
            "codigo_iso": pais.get("alpha3Code"),
            "capital": pais.get("capital"),
            "region": pais.get("region"),
            "subregion": pais.get("subregion"),
            "poblacion": pais.get("population"),
            "superficie_km2": pais.get("area"),
            "densidad_poblacional": pais.get("populationDensity")
        }

        registros.append(registro)

    # Convertimos los registros seleccionados
    # en una estructura tabular
    df_transformado = pd.DataFrame(registros)

    return df_transformado

### Lógica de la función de transformación

La función `transform_data()` recibe dos argumentos:

- `data`: conjunto de datos crudos extraídos desde la API.
- `region`: región que se desea seleccionar.

Dentro de la función se recorre cada observación mediante un ciclo `for`.

La condición `if` compara la región de cada registro con el valor
configurado en `REGION`. Cuando un registro no pertenece a la región
seleccionada, `continue` permite avanzar hacia la siguiente observación.

Para los registros que cumplen la condición se construye un nuevo
diccionario con las ocho variables seleccionadas previamente.

Finalmente, cada registro transformado se incorpora a una lista y
`pd.DataFrame()` convierte esa lista de diccionarios en una estructura
tabular.

Por lo tanto, esta función realiza:

**Filtrado → Selección de variables → Renombrado → Estructuración**

In [19]:
df = transform_data(
    raw_data,
    REGION
)

print("Transformación finalizada correctamente.")
print("Región seleccionada:", REGION)
print("Cantidad de observaciones:", df.shape[0])
print("Cantidad de variables:", df.shape[1])

Transformación finalizada correctamente.
Región seleccionada: Americas
Cantidad de observaciones: 57
Cantidad de variables: 8


In [20]:
df.head(10)

,pais,codigo_iso,capital,region,subregion,poblacion,superficie_km2,densidad_poblacional
0,Anguilla,AIA,The Valley,Americas,Caribbean,13452,91.0,147.82
1,Antigua and Barbuda,ATG,Saint John's,Americas,Caribbean,97928,442.0,221.56
2,Argentina,ARG,Buenos Aires,Americas,South America,45376763,2780400.0,16.32
3,Aruba,ABW,Oranjestad,Americas,Caribbean,106766,180.0,593.14
4,Bahamas,BHS,Nassau,Americas,Caribbean,393248,13943.0,28.20
5,Barbados,BRB,Bridgetown,Americas,Caribbean,287371,430.0,668.30
6,Belize,BLZ,Belmopan,Americas,Central America,397621,22966.0,17.31
7,Bermuda,BMU,Hamilton,Americas,Northern America,63903,54.0,1183.39
8,Bolivia (Plurinational State of),BOL,Sucre,Americas,South America,11673029,1098581.0,10.63
9,"Bonaire, Sint Eustatius and Saba",BES,Kralendijk,Americas,Caribbean,17408,294.0,59.21


In [21]:
print("Regiones presentes después del filtro:")
print(df["region"].unique())

Regiones presentes después del filtro:
['Americas']


In [22]:
filas, columnas = df.shape

print("Dimensiones del dataset transformado:")
print("Filas:", filas)
print("Columnas:", columnas)

Dimensiones del dataset transformado:
Filas: 57
Columnas: 8


### Validación de la transformación

Una vez ejecutada la transformación se verificaron las dimensiones del
DataFrame y las categorías presentes en la variable `region`.

El método `.unique()` permite comprobar qué valores distintos permanecen
en una columna después de aplicar el filtro.

De esta manera no se depende únicamente de una inspección visual de las
primeras filas, sino que se valida programáticamente que el dataset
resultante corresponde a la región seleccionada.

Por otra parte, `.shape` permite conocer las dimensiones de la estructura
resultante:

- `shape[0]`: cantidad de observaciones.
- `shape[1]`: cantidad de variables.

Estas comprobaciones permiten validar que la transformación haya
producido la estructura esperada antes de continuar con el análisis.

# 8. Análisis de estructura y calidad de los datos

Antes de realizar el análisis descriptivo y guardar el dataset se
evaluará su estructura y calidad.

Se revisarán:

- tipos de datos,
- valores no nulos,
- valores faltantes,
- porcentaje de valores faltantes,
- registros duplicados,
- duplicidad del identificador ISO.

Estos controles permiten detectar posibles problemas de calidad antes de
utilizar los datos para generar resultados.

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   pais                  57 non-null     object 
 1   codigo_iso            57 non-null     object 
 2   capital               56 non-null     object 
 3   region                57 non-null     object 
 4   subregion             57 non-null     object 
 5   poblacion             57 non-null     int64  
 6   superficie_km2        52 non-null     float64
 7   densidad_poblacional  52 non-null     float64
dtypes: float64(2), int64(1), object(5)
memory usage: 3.7+ KB


In [24]:
valores_nulos = df.isnull().sum()

porcentaje_nulos = (
    df.isnull().mean() * 100
).round(2)

calidad_datos = pd.DataFrame({
    "valores_nulos": valores_nulos,
    "porcentaje_nulos": porcentaje_nulos
})

calidad_datos

,valores_nulos,porcentaje_nulos
pais,0,0.00
codigo_iso,0,0.00
capital,1,1.75
region,0,0.00
subregion,0,0.00
poblacion,0,0.00
superficie_km2,5,8.77
densidad_poblacional,5,8.77


### Interpretación de la estructura y los valores faltantes

El dataset transformado contiene 57 observaciones y 8 variables.

Desde el punto de vista técnico, Pandas identificó variables de tipo
`object`, `int64` y `float64`. Esto permite diferenciar las variables
textuales de aquellas que pueden utilizarse directamente en operaciones
numéricas.

El análisis de completitud muestra que las variables `pais`,
`codigo_iso`, `region`, `subregion` y `poblacion` no presentan valores
faltantes.

Por otra parte:

- `capital` presenta 1 valor nulo, equivalente al 1.75 % de las
  observaciones.
- `superficie_km2` presenta 5 valores nulos, equivalentes al 8.77 %.
- `densidad_poblacional` presenta 5 valores nulos, equivalentes al
  8.77 %.

Los valores faltantes no serán eliminados de forma general, ya que
hacerlo implicaría descartar observaciones que todavía contienen
información útil en las demás variables.

En los análisis que requieran específicamente superficie o densidad
poblacional, los valores faltantes serán excluidos únicamente para
dicho cálculo.

In [25]:
registros_con_nulos = df[
    df.isnull().any(axis=1)
]

registros_con_nulos

,pais,codigo_iso,capital,region,subregion,poblacion,superficie_km2,densidad_poblacional
11,United States Minor Outlying Islands,UMI,None,Americas,Northern America,300,NaN,NaN
26,French Guiana,GUF,Cayenne,Americas,South America,254541,NaN,NaN
29,Guadeloupe,GLP,Basse-Terre,Americas,Caribbean,400132,NaN,NaN
35,Martinique,MTQ,Fort-de-France,Americas,Caribbean,378243,NaN,NaN
50,South Georgia and the South Sandwich Islands,SGS,King Edward Point,Americas,South America,30,NaN,NaN


### Validación básica del dataset

Como control posterior a la transformación se verificó la estructura
del DataFrame y la presencia de valores faltantes.

El objetivo de esta etapa no es realizar un proceso exhaustivo de
limpieza o análisis exploratorio, sino comprobar que la información
obtenida desde la API fue transformada correctamente a una estructura
tabular.

Se identificaron algunos valores faltantes en determinadas variables.
Estos valores se conservarán, ya que corresponden a la información
proporcionada por la fuente y no impiden cumplir con el objetivo de
extracción y persistencia de esta actividad.

# 8. Persistencia de los datos

Una vez finalizada la transformación, el dataset será almacenado en un
archivo CSV.

La persistencia permite conservar los datos obtenidos desde la API en un
formato estructurado que puede ser reutilizado posteriormente sin
depender de la respuesta almacenada temporalmente en memoria.

El nombre del archivo fue definido previamente mediante la variable de
entorno `OUTPUT_FILE`.

In [26]:
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Archivo '{OUTPUT_FILE}' generado correctamente."
)


Archivo 'data_extracted.csv' generado correctamente.


In [27]:
if os.path.exists(OUTPUT_FILE):

    print("Validación exitosa: el archivo fue creado.")

    df_validacion = pd.read_csv(OUTPUT_FILE)

    print(
        "Dimensiones del archivo:",
        df_validacion.shape
    )

else:

    print(
        "Error: el archivo no fue encontrado."
    )


Validación exitosa: el archivo fue creado.
Dimensiones del archivo: (57, 8)


In [28]:
df_validacion.head()

,pais,codigo_iso,capital,region,subregion,poblacion,superficie_km2,densidad_poblacional
0,Anguilla,AIA,The Valley,Americas,Caribbean,13452,91.0,147.82
1,Antigua and Barbuda,ATG,Saint John's,Americas,Caribbean,97928,442.0,221.56
2,Argentina,ARG,Buenos Aires,Americas,South America,45376763,2780400.0,16.32
3,Aruba,ABW,Oranjestad,Americas,Caribbean,106766,180.0,593.14
4,Bahamas,BHS,Nassau,Americas,Caribbean,393248,13943.0,28.20


### Validación del archivo de salida

Luego de generar el archivo CSV se verificó su existencia y se realizó
una nueva lectura mediante Pandas.

Esta comprobación permite confirmar que el proceso de persistencia fue
exitoso y que el dataset generado puede recuperarse posteriormente como
una fuente de datos estructurada.

# 9. Conclusión

En esta actividad se desarrolló un proceso completo de extracción,
transformación y persistencia de datos provenientes de una API pública.

Mediante una petición HTTP GET se obtuvo información en formato JSON.
Posteriormente, se seleccionaron las variables relevantes y se filtraron
los registros correspondientes a la región de América para construir
un dataset tabular mediante Pandas.

También se implementaron variables de entorno para separar la
configuración de la lógica del programa y manejo de excepciones para
controlar posibles fallas durante la comunicación con la API.

Finalmente, los datos transformados fueron almacenados en un archivo
CSV y se verificó que el archivo pudiera ser recuperado correctamente.

El flujo desarrollado puede resumirse como:

**API → GET → JSON → Transformación → DataFrame → CSV**

De esta manera se cumplen los objetivos de extracción, configuración,
manejo de errores, transformación y persistencia planteados para la
actividad.

# 10. Generación del script final

Durante el desarrollo se utilizaron diferentes celdas para explorar la
API, comprender la estructura de los datos y validar cada etapa.

A continuación se genera `extract_data.py`, que contiene únicamente la
lógica necesaria para ejecutar el proceso completo de forma independiente:

1. Carga de configuración.
2. Extracción mediante GET.
3. Manejo de errores.
4. Transformación.
5. Persistencia en CSV.

In [29]:
%%writefile extract_data.py
import os
import sys

import pandas as pd
import requests
from dotenv import load_dotenv


# 1. CARGA DE CONFIGURACIÓN

load_dotenv()

API_BASE_URL = os.getenv("API_BASE_URL")
REGION = os.getenv("REGION", "Americas")
OUTPUT_FILE = os.getenv("OUTPUT_FILE", "data_extracted.csv")
TIMEOUT = int(os.getenv("TIMEOUT", "10"))


# 2. EXTRACCIÓN DE DATOS

def extract_data():
    url = f"{API_BASE_URL}/countries"

    try:
        print("Iniciando extracción...")
        print("Endpoint:", url)

        response = requests.get(
            url,
            timeout=TIMEOUT
        )

        response.raise_for_status()

        data = response.json()

        print("Extracción realizada correctamente.")
        print("Registros recibidos:", len(data))

        return data

    except requests.exceptions.Timeout:
        print(
            "Error: la petición superó "
            "el tiempo máximo de espera."
        )

    except requests.exceptions.ConnectionError:
        print(
            "Error: no fue posible establecer "
            "conexión con la API."
        )

    except requests.exceptions.HTTPError as error:
        print(f"Error HTTP: {error}")

    except requests.exceptions.RequestException as error:
        print(f"Error durante la petición: {error}")

    except ValueError:
        print(
            "Error: la respuesta no contiene "
            "un JSON válido."
        )

    return None


# 3. TRANSFORMACIÓN

def transform_data(data, region):
    registros = []

    for pais in data:

        if pais.get("region") != region:
            continue

        registro = {
            "pais": pais.get("name"),
            "codigo_iso": pais.get("alpha3Code"),
            "capital": pais.get("capital"),
            "region": pais.get("region"),
            "subregion": pais.get("subregion"),
            "poblacion": pais.get("population"),
            "superficie_km2": pais.get("area"),
            "densidad_poblacional": pais.get(
                "populationDensity"
            )
        }

        registros.append(registro)

    return pd.DataFrame(registros)


# 4. PERSISTENCIA

def save_data(df):
    try:
        df.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Archivo '{OUTPUT_FILE}' "
            "generado correctamente."
        )

        return True

    except OSError as error:
        print(
            f"Error al guardar el archivo: {error}"
        )

        return False


# 5. FLUJO PRINCIPAL

def main():
    raw_data = extract_data()

    if raw_data is None:
        print("El proceso finalizó sin datos.")
        sys.exit(1)

    df = transform_data(
        raw_data,
        REGION
    )

    print(
        "Registros después de la transformación:",
        len(df)
    )

    if df.empty:
        print(
            "No se encontraron registros "
            "para la región seleccionada."
        )
        sys.exit(1)

    if not save_data(df):
        sys.exit(1)

    print("Proceso finalizado correctamente.")


if __name__ == "__main__":
    main()

Writing extract_data.py


In [30]:
!python extract_data.py

Iniciando extracción...
Endpoint: https://countries.dev/countries
Extracción realizada correctamente.
Registros recibidos: 250
Registros después de la transformación: 57
Archivo 'data_extracted.csv' generado correctamente.
Proceso finalizado correctamente.


In [31]:
%%writefile .env.example
API_BASE_URL=https://countries.dev
REGION=Americas
OUTPUT_FILE=data_extracted.csv
TIMEOUT=10

Writing .env.example


In [32]:
%%writefile .gitignore
# Variables de entorno
.env

# Entornos virtuales
.venv/
venv/

# Archivos temporales de Python
__pycache__/
*.pyc

# Editores
.vscode/
.idea/

# Sistema operativo
.DS_Store
Thumbs.db

Overwriting .gitignore


In [33]:
%%writefile requirements.txt
requests
pandas
python-dotenv

Overwriting requirements.txt


In [34]:
pip install -r requirements.txt

In [35]:
%%writefile README.md
# Pre-Entrega 4 - Data Science II

## Extracción y transformación de datos desde una API pública

### Descripción

Este proyecto fue desarrollado como parte de la Pre-Entrega 4 de Data
Science II.

El objetivo consiste en implementar un flujo de extracción, transformación
y persistencia de datos provenientes de una API REST pública utilizando
Python.

Se utiliza la API countries.dev para obtener información geográfica y
demográfica de países. A partir de la respuesta obtenida se seleccionan
los registros correspondientes a América y se construye un dataset
estructurado.

## Objetivo

Desarrollar un script reproducible que permita:

- Realizar una petición HTTP GET a una API pública.
- Recuperar y procesar una respuesta JSON.
- Administrar la configuración mediante variables de entorno.
- Implementar manejo de errores.
- Seleccionar y transformar variables.
- Guardar el resultado en formato CSV.

## Flujo del proyecto

API REST

↓

Petición HTTP GET

↓

Respuesta JSON

↓

Filtro geográfico

↓

Selección de variables

↓

DataFrame de Pandas

↓

data_extracted.csv

## Variables seleccionadas

El dataset final contiene las siguientes variables:

- `pais`: nombre del país o territorio.
- `codigo_iso`: código ISO de tres caracteres.
- `capital`: capital.
- `region`: región geográfica.
- `subregion`: subregión geográfica.
- `poblacion`: población.
- `superficie_km2`: superficie en kilómetros cuadrados.
- `densidad_poblacional`: densidad de población.

## Estructura del repositorio

```text
pre-entrega-4-api-paises/
│
├── extract_data.py
├── data_extracted.csv
├── README.md
├── requirements.txt
├── .env.example
└── .gitignore
```

## Requisitos

- Python 3
- requests
- pandas
- python-dotenv

## Instalación

Clonar el repositorio:

```bash
git clone URL_DEL_REPOSITORIO
```

Ingresar al proyecto:

```bash
cd pre-entrega-4-api-paises
```

Instalar las dependencias:

```bash
pip install -r requirements.txt
```

## Configuración

El proyecto utiliza variables de entorno para separar la configuración de
la lógica del programa.

Crear un archivo `.env` tomando como referencia `.env.example`.

Ejemplo:

```env
API_BASE_URL=https://countries.dev
REGION=Americas
OUTPUT_FILE=data_extracted.csv
TIMEOUT=10
```

La API seleccionada no requiere una API Key. En este proyecto, las
variables de entorno se utilizan para administrar parámetros de
configuración.

El archivo `.env` se encuentra incluido en `.gitignore` y no debe ser
publicado en el repositorio.

## Ejecución

Ejecutar:

```bash
python extract_data.py
```

El script realiza una petición GET a la API, procesa la respuesta JSON,
selecciona los registros correspondientes a la región configurada y genera
el archivo:

```text
data_extracted.csv
```

## Manejo de errores

El script implementa manejo de excepciones para diferentes situaciones:

- Timeout.
- Problemas de conexión.
- Errores HTTP.
- Errores generales durante la petición.
- Respuestas JSON inválidas.
- Errores durante la escritura del archivo.

También se utiliza `raise_for_status()` para detectar respuestas HTTP no
exitosas.

## Transformación

La respuesta original de la API contiene una cantidad mayor de atributos.

La etapa de transformación:

1. Filtra los registros correspondientes a `Americas`.
2. Selecciona únicamente las variables relevantes.
3. Renombra algunos atributos para facilitar su interpretación.
4. Convierte los registros en un DataFrame de Pandas.
5. Exporta el resultado en formato CSV.

## Archivo de salida

El resultado se almacena en:

```text
data_extracted.csv
```

Este archivo constituye una fuente estructurada que puede reutilizarse
posteriormente con Python, Excel, Power BI u otras herramientas de
análisis.

## Conclusión

El proyecto implementa un pipeline básico de datos que integra una fuente
externa mediante una API REST.

El flujo desarrollado puede resumirse como:

**Extracción → Transformación → Persistencia**

Además, incorpora buenas prácticas de configuración, manejo de errores y
documentación para facilitar su ejecución y reproducibilidad.

Writing README.md


In [36]:
import os

archivos_obligatorios = [
    "extract_data.py",
    "data_extracted.csv",
    "README.md",
    "requirements.txt",
    ".env.example",
    ".gitignore"
]

print("CONTROL DEL ENTREGABLE")
print("=" * 50)

for archivo in archivos_obligatorios:

    if os.path.exists(archivo):
        print("OK    ", archivo)

    else:
        print("FALTA ", archivo)

CONTROL DEL ENTREGABLE
OK     extract_data.py
OK     data_extracted.csv
OK     README.md
OK     requirements.txt
OK     .env.example
OK     .gitignore


In [37]:
!zip entrega_api.zip \
    extract_data.py \
    data_extracted.csv \
    README.md \
    requirements.txt \
    .env.example \
    .gitignore

  adding: extract_data.py (deflated 62%)
  adding: data_extracted.csv (deflated 52%)
  adding: README.md (deflated 56%)
  adding: requirements.txt (stored 0%)
  adding: .env.example (stored 0%)
  adding: .gitignore (deflated 26%)


In [38]:
from google.colab import files

files.download("entrega_api.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>